General Image-Level Features

mean_grey_value_image:

Description: The average pixel intensity across the entire image.
Interpretation: Indicates the overall brightness or "density" of the image. A higher value suggests a brighter image, while a lower value suggests a darker image. Can reflect material uniformity or global contrast in applications like quality control or visual inspection.

contrast:

Description: The standard deviation of pixel intensities.
Interpretation: Measures variability in pixel brightness. High contrast suggests a visually dynamic image with well-defined features, while low contrast indicates uniformity, which may highlight smooth surfaces or poor lighting conditions.

energy:

Description: The sum of squared probabilities from the pixel intensity histogram.
Interpretation: Represents image uniformity. High energy indicates a highly uniform texture, while low energy suggests more randomness, which could be linked to rough or uneven textures.

homogeneity:

Description: A measure of how similar pixel intensities are within the image.
Interpretation: Higher values reflect smoother textures or surfaces with little variation, while lower values indicate abrupt changes in intensity, like edges or fine details.

entropy:

Description: A measure of randomness in pixel intensity distribution.
Interpretation: Higher entropy indicates a more complex or diverse texture. Useful for detecting structural irregularities or chaotic patterns in materials.

variance:

Description: Variance in pixel intensities across the image.
Interpretation: Highlights overall dispersion in intensity values. Large variance reflects a wide range of brightness, potentially signifying mixed textures or materials.

skewness:

Description: Asymmetry of the pixel intensity distribution.
Interpretation: Positive skew suggests an image dominated by darker pixels, while negative skew suggests dominance by brighter pixels. Useful for distinguishing images with uneven lighting or unevenly distributed materials.

kurtosis:

Description: The "tailedness" of the pixel intensity distribution.
Interpretation: Higher kurtosis indicates more extreme intensity values (e.g., sharp edges or hotspots), while lower kurtosis suggests a more uniform spread.

sum_of_squares:

Description: Variance of the pixel intensity histogram.
Interpretation: Highlights the variability in intensity distribution across the image. Higher values indicate more distinct regions of varying brightness.

sum_average:

Description: Weighted average of intensity values based on the histogram.
Interpretation: Combines intensity and frequency information, reflecting the balance of brightness within the image.

sum_entropy:

Description: Entropy of the intensity sum distribution.
Interpretation: Similar to image entropy but calculated from summed intensities, providing insight into global randomness.

difference_entropy:

Description: Entropy of differences between adjacent intensity values.
Interpretation: Highlights local variability in pixel intensities, useful for detecting edge-rich or highly textured regions.

autocorrelation:

Description: A measure of repeating patterns in the pixel intensity histogram.
Interpretation: High autocorrelation reflects periodic patterns, while low autocorrelation suggests more randomness. Useful for identifying textures with regular structure.

Blob-Level Features

These features are computed based on detected blobs, or distinct regions of interest within the image.

num_blobs:

Description: The number of detected blobs.
Interpretation: Represents the density of distinct features or objects in the image. Can indicate fragmentation, clustering, or structural complexity in materials.

Blob Diameter Statistics (avg_diameter, std_diameter, min_diameter, max_diameter, median_diameter):

Description: Summary statistics for the diameters of detected blobs.
Interpretation: Reflects the size distribution of objects in the image. Large average diameters suggest the dominance of larger features, while higher standard deviation indicates variability in object size.

Blob Area Statistics (avg_area, std_area, min_area, max_area, median_area):

Description: Summary statistics for the areas of detected blobs.
Interpretation: Similar to diameter statistics but focused on total size. Useful for quantifying object coverage and identifying potential anomalies or outliers.

Blob Mean Grey Value Statistics (avg_mean_grey_blob, std_mean_grey_blob, min_mean_grey_blob, max_mean_grey_blob, median_mean_grey_blob):

Description: Summary statistics for average gray intensity within blobs.
Interpretation: Indicates the relative brightness or density within each blob. Can be used to identify differences in composition or material properties among features.

In [ ]:
# This block performs featurization on the full image without masking

import cv2
import numpy as np
import os
import pandas as pd
from scipy.stats import entropy, kurtosis, skew
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from itertools import product

# Directory containing images (edit as necessary)
image_dir = 'filepath'

# Get list of image files in the directory
image_files = [
    f for f in os.listdir(image_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))
]

# List to hold data for all images
data_list = []

# Loop over all images
for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    print(f"Processing {image_file}...")

    # Load the color image (in BGR format)
    image = cv2.imread(image_path)
    if image is None:
        print(f"Could not read {image_file}, skipping.")
        continue

    # Option to crop to a defined fraction of the image (for example, '70% ROI' = crop_ratio .7)
    height, width = image.shape[:2]
    crop_ratio = 1
    new_width = int(width * crop_ratio)
    new_height = int(height * crop_ratio)
    x_start = int((width - new_width) / 2)
    y_start = int((height - new_height) / 2)
    # Crop the image
    cropped_image = image[
        y_start:y_start + new_height, x_start:x_start + new_width
    ]

    # Convert the cropped image to grayscale for further analysis
    gray_image = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2GRAY)

    # Apply Gaussian Blur to smooth the image (adjust kernel size as needed)
    blurred_image = cv2.GaussianBlur(gray_image, (7, 7), 0)  # 5x5 kernel, sigmaX=0

    # Option to apply a simple contrast adjustment (no adjustment at alpha=1 and beta=0)
    alpha = 1  # Contrast control (1.0-3.0)
    beta = 0   # Brightness control (0-100)
    adjusted_image = cv2.convertScaleAbs(blurred_image, alpha=alpha, beta=beta)

    # Blob detection setup
    # Set up SimpleBlobDetector parameters.
    params = cv2.SimpleBlobDetector_Params()

    # Adjust thresholds
    params.minThreshold = 0
    params.maxThreshold = 255
    params.thresholdStep = 5  # Smaller step for finer detection

    # Filter by area
    params.filterByArea = True
    params.minArea = 35000      # Increased to ignore smaller blobs
    params.maxArea = 1e30      # Increased to include larger blobs

    # Enable filtering by circularity
    params.filterByCircularity = True
    params.minCircularity = 0.1  # Adjusted to include blobs that are not perfect circles

    # Enable filtering by convexity
    params.filterByConvexity = True
    params.minConvexity = 0.1  # Adjusted to include less convex blobs

    # Enable filtering by inertia (elongation)
    params.filterByInertia = True
    params.minInertiaRatio = 0.1  # Adjusted to include elongated blobs

    # Disable filtering by blob color
    params.filterByColor = False

    # Create a detector with the parameters
    detector = cv2.SimpleBlobDetector_create(params)

    # Detect blobs in the adjusted cropped image
    keypoints = detector.detect(adjusted_image)

    # Create a list to store mean grey values of blobs
    mean_grey_values = []

    # Collect diameters and areas of the detected blobs
    diameters = []
    areas = []

    for keypoint in keypoints:
        x, y = keypoint.pt  # Blob centroid
        diameter = keypoint.size  # Size of the blob
        radius = int(diameter / 2)

        # Create a mask for the blob
        mask = np.zeros_like(adjusted_image, dtype=np.uint8)
        cv2.circle(mask, (int(x), int(y)), radius, 255, thickness=-1)  # White filled circle

        # Calculate mean grey value within the blob
        mean_val = cv2.mean(adjusted_image, mask=mask)[0]
        mean_grey_values.append(mean_val)

        # Collect diameters and areas
        diameters.append(diameter)
        area = np.pi * (diameter / 2) ** 2  # Area of the blob
        areas.append(area)

    # Number of blobs detected
    num_blobs = len(keypoints)
    print(f"Number of blobs detected: {num_blobs}")

    # Convert lists to numpy arrays for statistical calculations
    diameters = np.array(diameters)
    areas = np.array(areas)
    mean_grey_values = np.array(mean_grey_values)

    # Initialize data dictionary for this image
    data = {'image file': image_file}

    # Calculate summary statistics if blobs are detected
    if num_blobs > 0:
        # Statistics for diameters
        data['avg_diameter'] = np.mean(diameters)
        data['std_diameter'] = np.std(diameters)
        data['min_diameter'] = np.min(diameters)
        data['max_diameter'] = np.max(diameters)
        data['median_diameter'] = np.median(diameters)

        # Statistics for areas
        data['avg_area'] = np.mean(areas)
        data['std_area'] = np.std(areas)
        data['min_area'] = np.min(areas)
        data['max_area'] = np.max(areas)
        data['median_area'] = np.median(areas)

        # Statistics for mean grey values within blobs
        data['avg_mean_grey_blob'] = np.mean(mean_grey_values)
        data['std_mean_grey_blob'] = np.std(mean_grey_values)
        data['min_mean_grey_blob'] = np.min(mean_grey_values)
        data['max_mean_grey_blob'] = np.max(mean_grey_values)
        data['median_mean_grey_blob'] = np.median(mean_grey_values)
    else:
        # Set to NaN if no blobs detected
        data['avg_diameter'] = np.nan
        data['std_diameter'] = np.nan
        data['min_diameter'] = np.nan
        data['max_diameter'] = np.nan
        data['median_diameter'] = np.nan

        data['avg_area'] = np.nan
        data['std_area'] = np.nan
        data['min_area'] = np.nan
        data['max_area'] = np.nan
        data['median_area'] = np.nan

        data['avg_mean_grey_blob'] = np.nan
        data['std_mean_grey_blob'] = np.nan
        data['min_mean_grey_blob'] = np.nan
        data['max_mean_grey_blob'] = np.nan
        data['median_mean_grey_blob'] = np.nan

    data['num_blobs'] = num_blobs

    # Texture analysis on the cropped image
    # Calculate the mean grey value (visual density) of the entire cropped image
    mean_grey_value = np.mean(adjusted_image)
    data['mean_grey_value_image'] = mean_grey_value

    # Calculate texture features using histogram and simple statistics
    hist = cv2.calcHist([adjusted_image], [0], None, [256], [0, 256])
    hist = hist.flatten()

    # Normalize the histogram to get probabilities
    hist_norm = hist / hist.sum()

    # Contrast: Standard deviation of the pixel values
    contrast = np.std(adjusted_image)
    data['contrast'] = contrast

    # Energy: Sum of squared elements in the normalized histogram
    energy = np.sum(hist_norm ** 2)
    data['energy'] = energy

    # Homogeneity: Calculated using the normalized histogram
    homogeneity = np.sum(hist_norm / (1 + np.arange(256)))
    data['homogeneity'] = homogeneity

    # Entropy: Measure of randomness in the image
    entropy_value = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))
    data['entropy'] = entropy_value

    # Variance: Variance of the pixel values
    variance = np.var(adjusted_image)
    data['variance'] = variance

    # Skewness: Measure of asymmetry of the pixel intensity distribution
    skewness = skew(adjusted_image.flatten())
    data['skewness'] = skewness

    # Kurtosis: Measure of the "tailedness" of the pixel intensity distribution
    kurt = kurtosis(adjusted_image.flatten())
    data['kurtosis'] = kurt

    # Sum of Squares (Variance of the histogram)
    mean_hist_value = np.mean(hist)
    sum_of_squares = np.sum((hist - mean_hist_value) ** 2)
    data['sum_of_squares'] = sum_of_squares

    # Sum Average: Average intensity based on the histogram
    sum_average = np.sum(np.arange(256) * hist_norm)
    data['sum_average'] = sum_average

    # Sum Entropy: Entropy of the sum distribution (same as image entropy here)
    sum_entropy = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))
    data['sum_entropy'] = sum_entropy

    # Difference Entropy: Entropy of the difference distribution
    diff_hist = np.abs(hist[:-1] - hist[1:])
    diff_hist_norm = diff_hist / diff_hist.sum()
    difference_entropy = -np.sum(diff_hist_norm * np.log2(diff_hist_norm + 1e-10))
    data['difference_entropy'] = difference_entropy

    # Autocorrelation: Measure of repeating patterns in the histogram
    autocorr = np.correlate(hist_norm, hist_norm, mode='full')
    autocorr_value = autocorr[autocorr.size // 2]
    data['autocorrelation'] = autocorr_value

    # Append the data for this image to the list
    data_list.append(data)

# Create a DataFrame from the collected data
df = pd.DataFrame(data_list)

# Display the DataFrame
print(df)

# Save to CSV in the same directory as the images
output_csv_path = os.path.join(image_dir, 'image_analysis_results.csv')
df.to_csv(output_csv_path, index=False)
print(f"\nResults saved to {output_csv_path}")

In [ ]:
# This block masks holes (threshold MGV >245) and performs featurization excluding holes/tears

import cv2
import numpy as np
import os
import pandas as pd
from scipy.stats import entropy, kurtosis, skew
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from itertools import product

# Directory containing images (edit as necessary)
image_dir = 'filepath'

# Get list of image files in the directory
image_files = [
    f for f in os.listdir(image_dir)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))
]

# List to hold data for all images
data_list = []

# Loop over all images
for image_file in image_files:
    image_path = os.path.join(image_dir, image_file)
    print(f"Processing {image_file}...")

    # Load the color image (in BGR format)
    image = cv2.imread(image_path)
    if image is None:
        print(f"Could not read {image_file}, skipping.")
        continue

    # Option to crop to a defined fraction of the image (for example, '70% ROI' = crop_ratio .7)
    height, width = image.shape[:2]
    crop_ratio = 1
    new_width = int(width * crop_ratio)
    new_height = int(height * crop_ratio)
    x_start = int((width - new_width) / 2)
    y_start = int((height - new_height) / 2)
    cropped_image = image[y_start:y_start + new_height, x_start:x_start + new_width]

    # Convert to grayscale
    gray_image = cv2.cvtColor(cropped_image, cv2.COLOR_BGR2GRAY)

    # Create a binary mask to exclude bright regions (holes/tears)
    _, exclusion_mask = cv2.threshold(gray_image, 245, 255, cv2.THRESH_BINARY)
    exclusion_mask_inv = cv2.bitwise_not(exclusion_mask)  # Mask of material only

    # Apply Gaussian blur to the grayscale image
    blurred_image = cv2.GaussianBlur(gray_image, (7, 7), 0)

    # Contrast and brightness adjustment
    alpha = 1  # Contrast control (1.0-3.0)
    beta = 0   # Brightness control (0-100)
    adjusted_image = cv2.convertScaleAbs(blurred_image, alpha=alpha, beta=beta)

    # Apply the inverse exclusion mask to remove holes/tears
    masked_image = cv2.bitwise_and(adjusted_image, adjusted_image, mask=exclusion_mask_inv)

    # Show verification of masking for the first image only
    if image_file == image_files[0]:
        import matplotlib.pyplot as plt

        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        axs[0].imshow(gray_image, cmap='gray')
        axs[0].set_title('Original Grayscale')
        axs[0].axis('off')

        axs[1].imshow(exclusion_mask_inv, cmap='gray')
        axs[1].set_title('Material Mask (Inverted)')
        axs[1].axis('off')

        axs[2].imshow(masked_image, cmap='gray')
        axs[2].set_title('Masked Image (Used in Analysis)')
        axs[2].axis('off')

        plt.tight_layout()
        plt.show()

    # Blob detection setup
    params = cv2.SimpleBlobDetector_Params()
    params.minThreshold = 0
    params.maxThreshold = 255
    params.thresholdStep = 5
    params.filterByArea = True
    params.minArea = 35000
    params.maxArea = 1e30
    params.filterByCircularity = True
    params.minCircularity = 0.1
    params.filterByConvexity = True
    params.minConvexity = 0.1
    params.filterByInertia = True
    params.minInertiaRatio = 0.1
    params.filterByColor = False
    detector = cv2.SimpleBlobDetector_create(params)

    # Detect blobs in the masked image
    keypoints = detector.detect(masked_image)

    # Create a list to store mean grey values of blobs
    mean_grey_values = []
    diameters = []
    areas = []

    for keypoint in keypoints:
        x, y = keypoint.pt
        diameter = keypoint.size
        radius = int(diameter / 2)

        mask = np.zeros_like(masked_image, dtype=np.uint8)
        cv2.circle(mask, (int(x), int(y)), radius, 255, thickness=-1)

        mean_val = cv2.mean(masked_image, mask=mask)[0]
        mean_grey_values.append(mean_val)

        diameters.append(diameter)
        area = np.pi * (diameter / 2) ** 2
        areas.append(area)

    num_blobs = len(keypoints)
    print(f"Number of blobs detected: {num_blobs}")

    diameters = np.array(diameters)
    areas = np.array(areas)
    mean_grey_values = np.array(mean_grey_values)

    data = {'image file': image_file}

    if num_blobs > 0:
        data['avg_diameter'] = np.mean(diameters)
        data['std_diameter'] = np.std(diameters)
        data['min_diameter'] = np.min(diameters)
        data['max_diameter'] = np.max(diameters)
        data['median_diameter'] = np.median(diameters)

        data['avg_area'] = np.mean(areas)
        data['std_area'] = np.std(areas)
        data['min_area'] = np.min(areas)
        data['max_area'] = np.max(areas)
        data['median_area'] = np.median(areas)

        data['avg_mean_grey_blob'] = np.mean(mean_grey_values)
        data['std_mean_grey_blob'] = np.std(mean_grey_values)
        data['min_mean_grey_blob'] = np.min(mean_grey_values)
        data['max_mean_grey_blob'] = np.max(mean_grey_values)
        data['median_mean_grey_blob'] = np.median(mean_grey_values)
    else:
        data['avg_diameter'] = np.nan
        data['std_diameter'] = np.nan
        data['min_diameter'] = np.nan
        data['max_diameter'] = np.nan
        data['median_diameter'] = np.nan

        data['avg_area'] = np.nan
        data['std_area'] = np.nan
        data['min_area'] = np.nan
        data['max_area'] = np.nan
        data['median_area'] = np.nan

        data['avg_mean_grey_blob'] = np.nan
        data['std_mean_grey_blob'] = np.nan
        data['min_mean_grey_blob'] = np.nan
        data['max_mean_grey_blob'] = np.nan
        data['median_mean_grey_blob'] = np.nan

    data['num_blobs'] = num_blobs

    # Global texture features on masked image
    valid_pixels = masked_image[exclusion_mask_inv > 0]
    data['mean_grey_value_image'] = np.mean(valid_pixels)

    hist = cv2.calcHist([masked_image], [0], exclusion_mask_inv, [256], [0, 256])
    hist = hist.flatten()
    hist_norm = hist / hist.sum()

    data['contrast'] = np.std(valid_pixels)
    data['energy'] = np.sum(hist_norm ** 2)
    data['homogeneity'] = np.sum(hist_norm / (1 + np.arange(256)))
    data['entropy'] = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))
    data['variance'] = np.var(valid_pixels)
    data['skewness'] = skew(valid_pixels.flatten())
    data['kurtosis'] = kurtosis(valid_pixels.flatten())

    mean_hist_value = np.mean(hist)
    data['sum_of_squares'] = np.sum((hist - mean_hist_value) ** 2)
    data['sum_average'] = np.sum(np.arange(256) * hist_norm)
    data['sum_entropy'] = -np.sum(hist_norm * np.log2(hist_norm + 1e-10))

    diff_hist = np.abs(hist[:-1] - hist[1:])
    diff_hist_norm = diff_hist / diff_hist.sum()
    data['difference_entropy'] = -np.sum(diff_hist_norm * np.log2(diff_hist_norm + 1e-10))

    autocorr = np.correlate(hist_norm, hist_norm, mode='full')
    data['autocorrelation'] = autocorr[autocorr.size // 2]

    data_list.append(data)

# Create and save DataFrame
df = pd.DataFrame(data_list)
print(df)
output_csv_path = os.path.join(image_dir, 'image_analysis_results_holes_excluded.csv')
df.to_csv(output_csv_path, index=False)
print(f"\nResults saved to {output_csv_path}")